# PyTorch Tutorial: Predicting Martensitic Transformation Temperature $M_s$ in NiTi-based Shape Memory Alloys

## Background

The martensitic transformation temperature $M_s$ in NiTi SMAs is exquisitely sensitive to composition:
- Every **0.1 at% increase in Ni** content suppresses $M_s$ by approximately **10 K**
- Ternary/quaternary alloying elements (Cu, Fe, Hf, Zr, Al) can shift $M_s$ by hundreds of degrees
- Experimental measurement is expensive — ML-based prediction has direct engineering value

## What This Notebook Covers

1. Generate a synthetic NiTi multi-component alloy dataset grounded in physical rules
2. Compare four models: **Linear Regression / MLP / ResNet / Physics-Constrained Net**
3. Demonstrate **Early Stopping** and **learning rate scheduling**
4. Use **MC Dropout** for uncertainty quantification (the neural network analog of GPR confidence intervals)
5. Use **SHAP** for feature importance analysis (the neural network analog of XGBoost Feature Gain)

---

## 0. Install Dependencies

In [ ]:
# 取消注释以安装依赖
# !pip install torch numpy matplotlib scikit-learn shap pandas

## 1. Imports & Device Selection

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# 固定随机种子，保证可复现性
torch.manual_seed(42)
np.random.seed(42)

# M5 Pro Mac 自动选择 MPS 加速；有 CUDA 则用 GPU；否则 CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"PyTorch version : {torch.__version__}")
print(f"Running on      : {device}")

PyTorch version : 2.11.0
Running on      : mps


## 2. Synthetic NiTi Dataset Generation

### Composition Space

Alloys are parameterized as **Ni$_{50-x}$Ti$_{50-y}$X$_y$** with the following features:

| Feature | Physical meaning |
|---------|------------------|
| `Ni_excess` | Ni deviation from stoichiometry (at%) — each +0.1% suppresses $M_s$ by ~10 K |
| `Cu` | Substitutes Ni — raises $M_s$, reduces thermal hysteresis |
| `Fe` | Substitutes Ni — strongly suppresses $M_s$ |
| `Hf` | Substitutes Ti — raises $M_s$ (high-temperature SMA) |
| `Zr` | Substitutes Ti — moderately raises $M_s$ |
| `Al` | Substitutes Ni — suppresses $M_s$ |
| `grain_size` | Grain size (μm) — affects transformation stress |
| `cold_work` | Cold work percentage — affects transformation temperature |

### Physical Model for $M_s$ (used to generate ground truth)

$$M_s = 60 - 100\,\text{Ni}_{\text{excess}} + 2\,\text{Cu} - 40\,\text{Fe} + 10\,\text{Hf} + 6\,\text{Zr} - 15\,\text{Al} - 0.3\,\text{CW} + \text{nonlinear terms} + \varepsilon$$

where $M_s^0 = 60\,°C$ is the reference value for stoichiometric NiTi.

In [2]:
def generate_NiTi_dataset(n_samples=1000):
    """
    生成合成 NiTi 多元合金数据集。
    成分约束：Ni端掺杂总量 < 10 at%，Ti端掺杂总量 < 14 at%
    """
    data = []

    while len(data) < n_samples:
        # 随机采样成分（at%）
        ni_excess  = np.random.uniform(0, 1.5)   # Ni 超化学计量比
        cu         = np.random.uniform(0, 8)      # Cu 替代 Ni
        fe         = np.random.uniform(0, 3)      # Fe 替代 Ni
        hf         = np.random.uniform(0, 12)     # Hf 替代 Ti
        zr         = np.random.uniform(0, 8)      # Zr 替代 Ti
        al         = np.random.uniform(0, 4)      # Al 替代 Ni

        # 成分约束：确保物理上合理
        if cu + fe + al > 10 or hf + zr > 14:
            continue

        # 工艺参数
        grain_size = np.random.lognormal(mean=2.5, sigma=0.8)  # 对数正态分布，μm
        cold_work  = np.random.uniform(0, 40)                  # 冷加工量 %

        # Ms 物理模型：线性主效应 + 非线性交互项
        Ms = (
            60
            - 100 * ni_excess           # Ni 超量：强烈降低 Ms
            +   2 * cu                  # Cu：适度升高
            -  40 * fe                  # Fe：强烈降低
            +  10 * hf                  # Hf：升高（高温 SMA 设计）
            +   6 * zr                  # Zr：适度升高
            -  15 * al                  # Al：降低
            - 0.3 * cold_work           # 冷加工：轻微降低
            + 0.5 * np.log(grain_size + 1)   # 晶粒尺寸：对数关系（Hall-Petch 类比）
            + 3.0 * cu * hf / 10        # Cu-Hf 协同效应（非线性交互）
            - 5.0 * fe * ni_excess      # Fe-Ni超量 协同降低（非线性交互）
        )

        # 加入实验测量噪声（±5°C 量级）
        Ms += np.random.normal(0, 5)

        data.append({
            'Ni_excess':  ni_excess,
            'Cu':         cu,
            'Fe':         fe,
            'Hf':         hf,
            'Zr':         zr,
            'Al':         al,
            'grain_size': grain_size,
            'cold_work':  cold_work,
            'Ms':         Ms
        })

    return pd.DataFrame(data)


df = generate_NiTi_dataset(n_samples=1000)
print(f"Dataset size: {len(df)} alloys")
print(f"\nDescriptive statistics:")
print(df.describe().round(2))

Dataset size: 1000 alloys

Descriptive statistics:
       Ni_excess       Cu       Fe       Hf       Zr       Al  grain_size  \
count    1000.00  1000.00  1000.00  1000.00  1000.00  1000.00     1000.00   
mean        0.76     3.21     1.38     5.13     3.59     1.83       17.00   
std         0.44     1.96     0.85     3.03     2.27     1.14       15.93   
min         0.00     0.00     0.00     0.00     0.01     0.00        1.02   
25%         0.36     1.61     0.62     2.65     1.68     0.79        7.10   
50%         0.77     3.14     1.38     5.12     3.35     1.79       12.36   
75%         1.16     4.68     2.08     7.24     5.39     2.78       21.26   
max         1.50     7.99     3.00    11.95     7.95     3.99      152.36   

       cold_work       Ms  
count    1000.00  1000.00  
mean       19.97   -24.26  
std        11.38    71.19  
min         0.03  -230.28  
25%        10.30   -71.81  
50%        19.93   -22.39  
75%        30.23    26.52  
max        39.97   181.65  


In [ ]:
# 可视化各成分对 Ms 的影响（散点图 + 趋势线）
feature_cols = ['Ni_excess', 'Cu', 'Fe', 'Hf', 'Zr', 'Al', 'grain_size', 'cold_work']
colors = ['#E07B54', '#5B8DB8', '#6AAF6A', '#B07CC6',
          '#E8A838', '#D45F5F', '#4EAAB0', '#7D7D7D']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, feat, color in zip(axes.ravel(), feature_cols, colors):
    ax.scatter(df[feat], df['Ms'], alpha=0.3, s=8, color=color)
    # 一次多项式趋势线
    z = np.polyfit(df[feat], df['Ms'], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.6, linewidth=1.5)
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('$M_s$ (°C)', fontsize=10)
    ax.set_title(f'{feat} vs $M_s$', fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Effect of Each Feature on $M_s$ in NiTi Alloys', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Ms range : {df['Ms'].min():.1f} ~ {df['Ms'].max():.1f} °C")
print(f"Ms mean  : {df['Ms'].mean():.1f} ± {df['Ms'].std():.1f} °C")

## 3. Data Preprocessing

In [ ]:
X = df[feature_cols].values
y = df['Ms'].values

# 80/20 分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 标准化特征：只在训练集上 fit，测试集只做 transform（避免数据泄露）
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

# 标准化目标值（有助于神经网络训练稳定）
scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

# 转换为 PyTorch Tensor 并移到目标设备
X_train_t = torch.FloatTensor(X_train_sc).to(device)
X_test_t  = torch.FloatTensor(X_test_sc).to(device)
y_train_t = torch.FloatTensor(y_train_sc).to(device)
y_test_t  = torch.FloatTensor(y_test_sc).to(device)

# DataLoader 负责自动分批（batch_size=32）和随机打乱
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32, shuffle=True
)

input_dim = X_train_t.shape[1]
print(f"Input features : {input_dim}")
print(f"Train set      : {X_train_t.shape}")
print(f"Test set       : {X_test_t.shape}")

## 4. Model Definitions

### New concept: Residual Connection

Instead of learning $f(x)$ directly, a residual block learns the **residual** $f(x) - x$:

```
Plain network  :  x → [Linear → ReLU → Linear] → y
Residual block :  x → [Linear → ReLU → Linear] → (+x) → y
                                                    ↑
                                               skip connection
```

The skip connection provides a gradient highway through the network, mitigating vanishing gradients in deep architectures.

### New concept: Physics-Constrained Network

Hard-code known physics (e.g., Ni excess linearly suppresses $M_s$) into the network architecture, leaving only the **nonlinear residual** for the data-driven component to learn.

In [ ]:
# ============================================================
# Model 1: Linear Baseline
# ============================================================
class LinearBaseline(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # 单层线性变换，等价于普通线性回归

    def forward(self, x):
        return self.linear(x).squeeze()


# ============================================================
# Model 2: MLP with BatchNorm
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 128, 64], dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),   # 批归一化：对每个 batch 做归一化，稳定训练
                nn.ReLU(),
                nn.Dropout(dropout)  # Dropout：训练时随机丢弃神经元，防止过拟合
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # 输出层：无激活函数（回归任务）
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze()


# ============================================================
# Model 3: ResNet-style Network
# ============================================================
class ResidualBlock(nn.Module):
    """两层线性变换 + 跳跃连接（skip connection）"""
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # 关键：输出 = 输入 + 残差，梯度可以直接通过跳跃连接反传
        return self.relu(x + self.block(x))


class ResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, n_blocks=3):
        super().__init__()
        self.embed  = nn.Linear(input_dim, hidden_dim)  # 将输入投影到隐藏维度
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim) for _ in range(n_blocks)]
        )
        self.head = nn.Linear(hidden_dim, 1)  # 输出头

    def forward(self, x):
        x = torch.relu(self.embed(x))  # 初始嵌入
        x = self.blocks(x)             # 经过多个残差块
        return self.head(x).squeeze()


# ============================================================
# Model 4: Physics-Constrained Network
# ============================================================
class PhysicsConstrainedNet(nn.Module):
    """
    将物理先验硬编码进网络架构：
      - 物理线性项：对应已知成分效应（系数可学习，但初始化为物理值）
      - 数据驱动项：神经网络学习非线性残差（交互效应等）
      - alpha：可学习的混合权重，控制物理项 vs 数据项的贡献比例

    特征顺序: [Ni_excess, Cu, Fe, Hf, Zr, Al, grain_size, cold_work]
    """
    def __init__(self, input_dim):
        super().__init__()

        # 物理先验系数（标准化空间中的近似值，可训练）
        self.physics_weights = nn.Parameter(
            torch.tensor([-1.5, 0.2, -0.8, 0.3, 0.15, -0.4, 0.05, -0.1],
                         dtype=torch.float32)
        )

        # 数据驱动子网：专门学习非线性交互项
        self.residual_net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # 混合权重 alpha（用 sigmoid 约束在 [0,1]）
        # 初始 alpha=0.7 表示 70% 来自物理项，30% 来自数据驱动项
        self.alpha = nn.Parameter(torch.tensor(0.7))

    def forward(self, x):
        physics_pred  = (x * self.physics_weights).sum(dim=1)  # 物理线性项
        residual_pred = self.residual_net(x).squeeze()         # 数据驱动残差
        alpha = torch.sigmoid(self.alpha)                      # 确保在 [0,1]
        return alpha * physics_pred + (1 - alpha) * residual_pred


def count_params(model):
    """统计模型可训练参数量"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Model parameter counts:")
for name, model in [
    ('LinearBaseline',        LinearBaseline(input_dim)),
    ('MLP',                   MLP(input_dim)),
    ('ResNet',                ResNet(input_dim)),
    ('PhysicsConstrainedNet', PhysicsConstrainedNet(input_dim)),
]:
    print(f"  {name:<28} {count_params(model):>6,} params")

## 5. Training with Early Stopping

**Early Stopping**: halt training when the validation loss stops improving for $N$ consecutive epochs. This prevents overfitting more gracefully than a fixed epoch budget, and saves compute.

**Cosine Annealing LR**: the learning rate follows a cosine curve from `lr` down to near zero, allowing the optimizer to take large steps early and fine-tune later — without the abrupt drops of step-based schedulers.

In [ ]:
class EarlyStopping:
    """早停器：监控验证 loss，自动保存最优模型权重"""

    def __init__(self, patience=20, min_delta=1e-4):
        self.patience      = patience    # 容忍多少个 epoch 没有改善
        self.min_delta     = min_delta   # 改善的最小阈值
        self.counter       = 0
        self.best_loss     = float('inf')
        self.best_state    = None        # 存储最优权重
        self.stopped_epoch = 0

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            # 有改善：更新最优状态
            self.best_loss  = val_loss
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter    = 0
        else:
            # 没有改善：计数器 +1
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped_epoch = epoch
                return True  # 触发早停
        return False

    def restore_best(self, model):
        """恢复到历史最优权重"""
        if self.best_state:
            model.load_state_dict(self.best_state)


def train_model(model, train_loader, X_val, y_val,
                n_epochs=500, lr=1e-3, patience=25):
    model     = model.to(device)
    # AdamW = Adam + 权重衰减（L2正则化的解耦版本，效果更好）
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    # 余弦退火：lr 从初始值余弦衰减到接近0
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    early_stop = EarlyStopping(patience=patience)

    train_losses, val_losses = [], []

    for epoch in range(n_epochs):
        # ---- 训练阶段 ----
        model.train()   # 开启 Dropout 和 BatchNorm 的训练模式
        epoch_loss = 0

        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            # 梯度裁剪：防止梯度爆炸（尤其对 ResNet 有帮助）
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / len(train_loader)

        # ---- 验证阶段 ----
        model.eval()    # 关闭 Dropout，BatchNorm 使用全局统计
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()

        scheduler.step()
        train_losses.append(avg_train)
        val_losses.append(val_loss)

        # 检查是否触发早停
        if early_stop(val_loss, model, epoch):
            print(f"  Early stopping at epoch {epoch+1}  "
                  f"(best val loss: {early_stop.best_loss:.4f})")
            break

    early_stop.restore_best(model)  # 加载历史最优权重
    return train_losses, val_losses, early_stop.stopped_epoch


def evaluate(model, X_test, y_test_sc, scaler_y):
    """在原始尺度（°C）上计算 R² 和 MAE"""
    model.eval()
    with torch.no_grad():
        y_pred_sc = model(X_test).cpu().numpy()
    # 反标准化回原始 °C 尺度
    y_pred = scaler_y.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()
    y_true = scaler_y.inverse_transform(y_test_sc.cpu().numpy().reshape(-1, 1)).ravel()
    return r2_score(y_true, y_pred), mean_absolute_error(y_true, y_pred), y_true, y_pred

In [ ]:
models_to_train = {
    'Linear Baseline':         LinearBaseline(input_dim),
    'MLP':                     MLP(input_dim),
    'ResNet':                  ResNet(input_dim),
    'Physics-Constrained Net': PhysicsConstrainedNet(input_dim),
}

results = {}

for name, model in models_to_train.items():
    print(f"\n{'='*52}")
    print(f"  Training: {name}")
    print('='*52)

    train_losses, val_losses, stopped = train_model(
        model, train_loader, X_test_t, y_test_t,
        n_epochs=500, lr=1e-3, patience=25
    )

    r2, mae, y_true, y_pred = evaluate(model, X_test_t, y_test_t, scaler_y)
    print(f"  Test R² = {r2:.4f}  |  MAE = {mae:.2f} °C")

    results[name] = {
        'model':       model,
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'r2':  r2,
        'mae': mae,
        'y_true': y_true,
        'y_pred': y_pred
    }

## 6. Results Visualization

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
colors = ['#7D7D7D', '#E07B54', '#5B8DB8', '#6AAF6A']

for col, (name, res) in enumerate(results.items()):
    color = colors[col]

    # 上行：Loss 曲线（对数坐标）
    ax = axes[0, col]
    ax.plot(res['train_losses'], label='Train', color=color, alpha=0.8)
    ax.plot(res['val_losses'],   label='Val',   color=color,
            linestyle='--', alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'{name}\nLoss Curve', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_yscale('log')  # 对数纵轴使收敛过程更清晰

    # 下行：预测 vs 真实散点图
    ax = axes[1, col]
    y_true, y_pred = res['y_true'], res['y_pred']
    ax.scatter(y_true, y_pred, alpha=0.4, color=color, s=15)
    lim = [min(y_true.min(), y_pred.min()) - 10,
           max(y_true.max(), y_pred.max()) + 10]
    ax.plot(lim, lim, 'r--', alpha=0.7, label='Ideal')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual $M_s$ (°C)')
    ax.set_ylabel('Predicted $M_s$ (°C)')
    ax.set_title(f'R² = {res["r2"]:.3f}  |  MAE = {res["mae"]:.1f} °C',
                 fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('NiTi $M_s$ Prediction — Four Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

# 汇总表格
print("\nModel Performance Summary:")
print(f"{'Model':<28} {'R²':>8} {'MAE (°C)':>10} {'Params':>8}")
print("-" * 58)
for name, res in results.items():
    n = count_params(res['model'])
    print(f"{name:<28} {res['r2']:>8.4f} {res['mae']:>10.2f} {n:>8,}")

## 7. Uncertainty Quantification via MC Dropout

GPR provides calibrated uncertainty estimates natively through its posterior variance. Neural networks require additional techniques. **MC Dropout** is the simplest approach:

- Keep Dropout **active at inference time** (i.e., call `model.train()` instead of `model.eval()`)
- Run $T$ stochastic forward passes through the same input
- The variance across passes approximates the **epistemic uncertainty** of the model

This is equivalent to variational inference over the model weights (Gal & Ghahramani, 2016).

In [ ]:
def mc_dropout_predict(model, X, n_samples=200):
    """
    MC Dropout 推理：
    - model.train() 保持 Dropout 激活
    - 多次前向传播，用输出方差估计认知不确定性
    返回：均值预测、标准差（不确定性代理）
    """
    model.train()   # 关键：train 模式才能保持 Dropout 开启
    predictions = []

    with torch.no_grad():
        for _ in range(n_samples):
            pred = model(X).cpu().numpy()
            predictions.append(pred)

    predictions = np.array(predictions)     # shape: [n_samples, n_test]
    return predictions.mean(axis=0), predictions.std(axis=0)


# 用 MLP 做 MC Dropout 演示
mlp_model = results['MLP']['model']
mean_sc, std_sc = mc_dropout_predict(mlp_model, X_test_t, n_samples=200)

# 反标准化回 °C 尺度
mean_pred = scaler_y.inverse_transform(mean_sc.reshape(-1, 1)).ravel()
std_pred  = std_sc * scaler_y.scale_[0]   # 标准差用 scale 还原（不加 mean）
y_true    = results['MLP']['y_true']

# 按真实值排序，便于可视化
sort_idx    = np.argsort(y_true)
y_true_s    = y_true[sort_idx]
mean_pred_s = mean_pred[sort_idx]
std_pred_s  = std_pred[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：置信区间带（对应 GPR 的 ±2σ 带）
ax = axes[0]
x_idx = np.arange(len(y_true_s))
ax.fill_between(x_idx,
                mean_pred_s - 2 * std_pred_s,
                mean_pred_s + 2 * std_pred_s,
                alpha=0.3, color='#5B8DB8', label='95% CI (MC Dropout)')
ax.plot(x_idx, mean_pred_s, color='#5B8DB8', linewidth=1.5, label='MC Mean')
ax.scatter(x_idx, y_true_s, color='#E07B54', s=10, alpha=0.6, label='True $M_s$')
ax.set_xlabel('Sample index (sorted by true $M_s$)')
ax.set_ylabel('$M_s$ (°C)')
ax.set_title('MC Dropout Uncertainty Bands\n(Neural network analog of GPR confidence intervals)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 右图：不确定性 vs 实际误差（理想情况下应正相关）
ax = axes[1]
errors = np.abs(mean_pred - y_true)
ax.scatter(std_pred, errors, alpha=0.4, s=15, color='#6AAF6A')
ax.set_xlabel('MC Dropout std (uncertainty proxy) (°C)')
ax.set_ylabel('Absolute prediction error (°C)')
ax.set_title('Uncertainty vs Actual Error\n(Should be positively correlated for well-calibrated models)')
corr = np.corrcoef(std_pred, errors)[0, 1]
ax.text(0.05, 0.95, f'Pearson r = {corr:.3f}',
        transform=ax.transAxes, fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Uncertainty–error correlation: {corr:.3f}")
print("(Closer to 1.0 → better-calibrated uncertainty estimates)")

## 8. Feature Importance via SHAP

XGBoost provides feature importance natively through split gain. Neural networks require post-hoc methods. **SHAP (SHapley Additive exPlanations)** is grounded in cooperative game theory:

> The SHAP value of feature $i$ for sample $x$ is its average marginal contribution across all possible subsets of features — the fairest possible attribution.

A positive SHAP value means the feature **raised** the prediction relative to the baseline; negative means it **lowered** it.

In [ ]:
try:
    import shap

    # SHAP 需要 CPU numpy 接口
    mlp_cpu = results['MLP']['model'].to('cpu')
    mlp_cpu.eval()

    def model_predict(x_np):
        """将 numpy 输入包装为 PyTorch 推理（SHAP 要求的接口）"""
        with torch.no_grad():
            return mlp_cpu(torch.FloatTensor(x_np)).numpy()

    # 用训练集前100个样本作为 SHAP 背景分布（KernelSHAP 的参考基准）
    background  = X_train_sc[:100]
    explainer   = shap.KernelExplainer(model_predict, background)

    # 计算前50个测试样本的 SHAP 值（KernelSHAP 较慢，nsamples 控制精度/速度权衡）
    print("Computing SHAP values (may take ~30-60 s)...")
    shap_values = explainer.shap_values(X_test_sc[:50], nsamples=200)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 左图：全局重要性（平均绝对 SHAP 值，对应 XGBoost Gain）
    ax = axes[0]
    mean_shap  = np.abs(shap_values).mean(axis=0)
    sorted_idx = np.argsort(mean_shap)
    # 根据平均效应方向着色
    bar_colors = ['#E07B54' if shap_values[:, i].mean() > 0 else '#5B8DB8'
                  for i in sorted_idx]
    ax.barh([feature_cols[i] for i in sorted_idx],
            mean_shap[sorted_idx], color=bar_colors)
    ax.set_xlabel('Mean |SHAP value| (°C)')
    ax.set_title('Global Feature Importance (SHAP)\nOrange = raises $M_s$, Blue = lowers $M_s$')
    ax.grid(alpha=0.3, axis='x')

    # 右图：Beeswarm — 每个样本的 SHAP 值，颜色代表特征值大小
    ax = axes[1]
    for i, feat in enumerate(feature_cols):
        sc = ax.scatter(
            shap_values[:, i],
            [feat] * shap_values.shape[0],
            c=X_test_sc[:50, i],   # 颜色 = 标准化后的特征值
            cmap='coolwarm',
            alpha=0.6, s=20
        )
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('SHAP value (impact on $M_s$ prediction, °C)')
    ax.set_title('SHAP Beeswarm Plot\nRed = high feature value, Blue = low feature value')
    plt.colorbar(sc, ax=ax, label='Feature value (standardized)')
    ax.grid(alpha=0.3, axis='x')

    plt.tight_layout()
    plt.show()

except ImportError:
    print("SHAP not installed — falling back to gradient-based importance.")
    print("Install with: pip install shap")
    print()

    # 用输入梯度近似特征重要性（快速替代方案）
    mlp_model = results['MLP']['model'].to('cpu')
    X_cpu = X_test_t.cpu().requires_grad_(True)
    mlp_model.eval()
    output = mlp_model(X_cpu).sum()
    output.backward()

    # 梯度绝对值的均值作为重要性代理
    grad_importance = X_cpu.grad.abs().mean(dim=0).detach().numpy()
    sorted_idx = np.argsort(grad_importance)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh([feature_cols[i] for i in sorted_idx],
            grad_importance[sorted_idx], color='#5B8DB8')
    ax.set_xlabel('Mean |gradient| (approximate feature importance)')
    ax.set_title('Gradient-based Feature Importance\n(Approximate substitute for SHAP)')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## 9. Concept Summary

### New concepts introduced in this notebook

| Concept | Role | Classical ML analog |
|---------|------|---------------------|
| `BatchNorm1d` | Normalizes activations per batch — stabilizes training | Feature standardization |
| `Dropout` | Randomly zeroes neurons during training — regularization | Ridge / Lasso penalty |
| `AdamW` | Adam with decoupled weight decay | L2 regularization |
| `CosineAnnealingLR` | Smooth LR decay following a cosine curve | Fixed learning rate |
| `clip_grad_norm_` | Prevents gradient explosion in deep nets | — |
| Residual connection | Skip connections mitigate vanishing gradients | — |
| Early stopping | Halt at best generalization, save compute | CV-based hyperparameter selection |
| MC Dropout | Approximate Bayesian uncertainty at inference | GPR posterior variance |
| SHAP | Post-hoc feature attribution via Shapley values | XGBoost Feature Gain |

### Direct correspondence with your GPR + XGBoost work

```
Your existing work              This notebook
──────────────────────────────────────────────────────
GPR posterior variance    ↔    MC Dropout std
XGBoost Feature Gain      ↔    SHAP global importance
Matérn kernel smoothness  ↔    ResNet inductive bias
Physics-informed features ↔    PhysicsConstrainedNet
Marginal likelihood opt.  ↔    AdamW + early stopping
```

### Suggested next steps

1. **Plug in your real dataset** (Xue 2017 CSV) — replace the synthetic data generation cell and re-run everything
2. **Hyperparameter search** with `optuna`: automate the search over network width, depth, dropout rate, and learning rate
3. **Transfer learning**: pre-train on a large materials database (AFLOW, Materials Project), then fine-tune on your small experimental dataset